In [8]:
%load_ext autoreload
%autoreload 2


import importlib
import random

### My libraries (not pip)

import Mathlib
import Activation
import Loss
import Optimizer
import Accuracy
import Neuron
import Layer
import Batch

importlib.reload(Mathlib)
importlib.reload(Activation)
importlib.reload(Loss)
importlib.reload(Optimizer)
importlib.reload(Accuracy)
importlib.reload(Neuron)
importlib.reload(Layer)
importlib.reload(Batch)

from Neuron import Neuron
from Layer import Layer
from Batch import Batch


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


#### Go to `proofs_math/ActivationFuncs` for all Activation function proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/ActivationFuncs/SoftmaxDerivative.jpg?version=1"
         alt="Softmax Proof"
         width="500">
  </div>

  <div style="display: flex; flex-direction: column; gap: 20px;">
    <img src="./proofs_math/ActivationFuncs/ReLUDerivative.jpg?version=1"
         alt="ReLU Proof"
         width="500">
  </div>

</div>



#### Go to `proofs_math/Loss` for all Loss proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/Loss/SoftmaxCrossEntropyDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
  <div style="display: flex; flex-direction: column; gap: 20px;">
    <img src="./proofs_math/Loss/LossDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
</div>

#### Go to `proofs_math/Neuron` for all Neuron proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/Neuron/NeuronDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
</div>



## Function that will be used to create datasets

In [9]:
def createInputs(inputCount=10):
    return([Mathlib.randomNumber() for _ in range(inputCount)])

## Functions to run a forward and backward pass on a super simple network with two layer totaling 9 neurons

In [10]:
random.seed(1)

inputsCount = 5
target = 2
inputs = createInputs(inputsCount)

layer1NeuronCount = 4
layer2NeuronCount = 5

layer1 = Layer(inputsCount, layer1NeuronCount, activationFunc=Activation.ReLU)         #< weights/biases determine neuron count
layer2 = Layer(layer1NeuronCount, layer2NeuronCount, activationFunc=Activation.ReLU)   #< weights/biases determine neuron count

accuracyCalculator = Accuracy.Accuracy_hard()

def stepForward():
    layer1Output = layer1.forward(inputs)
    layer2Output = layer2.forward(layer1Output)
    softmaxOutput = Activation.Softmax.forward(layer2Output)
    error = Loss.Entropy.forwardSparse(softmaxOutput, target)
    accuracy = accuracyCalculator.epoch(softmaxOutput, target)    #< we want to maximize the 2nd output
    return(layer1Output, layer2Output, softmaxOutput, error, accuracy)

def stepBackward(softmaxOutput):
    d_error  = Loss.Entropy.backwardSparse(softmaxOutput, target)
    d_softmax = Activation.Softmax.backward(softmaxOutput)
    d_crossEntropy = Mathlib.dotVectorMatrix(d_error, d_softmax) #< not that we can calculate the cross entropy function much simpler using the shortcut backwardCrossEntropy, but I choose to take the long way in this example because its easier to understand
    d_layer2 = layer2.backward(d_crossEntropy)
    d_layer1 = layer1.backward(d_layer2)
    return(d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1)

def printForwardSummary(layer1Output, layer2Output, softmaxOutput, error, accuracy):
    print(f"INPUTS: {inputs}\n    This layer should have {inputsCount} inputs")
    print(f"WEIGHTS: {layer1.getWeights()}\n    this layer should have {layer1NeuronCount} weights with {inputsCount} values in them")
    print(f"BIASES: {layer1.getBiases()}\n    this layer should have {layer1NeuronCount} biases")
    print(f"RESULT: {layer1Output}\n     this layer should have {layer1NeuronCount} outputs")
    print()
    print(f"WEIGHTS: {layer1.getWeights()}\n    this layer should have {layer2NeuronCount} weights with {len(layer1Output)} values in them")
    print(f"BIASES: {layer1.getBiases()}\n    this layer should have {layer2NeuronCount} biases")
    print(f"RESULT: {layer2Output}\n     this layer should have {layer2NeuronCount} outputs")
    print("SOFTMAX: ", softmaxOutput)
    print("TARGET: ", target)
    print(f"ERROR: {error} should be between 0 and 16.12")
    print("ACCURACY: ", accuracy)

def printBackwardSummary(d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1):
    print("d_ERROR: ", d_error)
    print(f"d_SOFTMAX: {d_softmax}\n    this layer should be {layer2NeuronCount}x{len(d_softmax)}")
    print(f"d_CROSS_ENTROPY: {d_crossEntropy}\n    this layer should have {layer2NeuronCount} outputs")
    print("d_LAYER 2: ", d_layer2)
    print("d_LAYER 1: ", d_layer1)

layer1Output, layer2Output, softmaxOutput, error, accuracy = stepForward()
printForwardSummary(layer1Output, layer2Output, softmaxOutput, error, accuracy)
d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1 = stepBackward(softmaxOutput)
printBackwardSummary(d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1)

INPUTS: [-7.31, 6.95, 5.28, -4.9, -0.09]
    This layer should have 5 inputs
WEIGHTS: [[-0.09, 0.27, 0.52, -0.73, -0.84], [0.6, -0.12, 0.47, -0.89, -0.1], [0.4, -0.49, 0.8, 0.72, -0.84], [-0.85, 0.07, 0.79, -0.21, -0.51]]
    this layer should have 4 weights with 5 values in them
BIASES: [0.0, 0.0, 0.0, 0.0]
    this layer should have 4 biases
RESULT: [8.932599999999999, 1.6316000000000015, 0.0, 11.946100000000001]
     this layer should have 4 outputs

WEIGHTS: [[-0.09, 0.27, 0.52, -0.73, -0.84], [0.6, -0.12, 0.47, -0.89, -0.1], [0.4, -0.49, 0.8, 0.72, -0.84], [-0.85, 0.07, 0.79, -0.21, -0.51]]
    this layer should have 5 weights with 4 values in them
BIASES: [0.0, 0.0, 0.0, 0.0]
    this layer should have 5 biases
RESULT: [0.0, 0.0, 6.723468000000001, 13.266073000000002, 10.44774]
     this layer should have 5 outputs
SOFTMAX:  [1.6324545230493332e-06, 1.6324545230493332e-06, 0.0013577072690506196, 0.9423742093689359, 0.056264818452967394]
TARGET:  2
ERROR: 6.601957833427744 should 

### Optimizing the neural network

In [11]:
optimizer = Optimizer.Optimizer_SGD(learning_rate=0.0001)
for i in range(5000):
    layer1Output, layer2Output, softmaxOutput, error, accuracy = stepForward()
    d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1 = stepBackward(softmaxOutput)
    optimizer.update(layer2)
    optimizer.update(layer1)
    print(f"Epoch {i+1}: Error: {error}, accuracy: {accuracy}")

Epoch 1: Error: 6.601957833427744, accuracy: 0.0
Epoch 2: Error: 6.550970740358418, accuracy: 0.0
Epoch 3: Error: 6.500165955078458, accuracy: 0.0
Epoch 4: Error: 6.449544711689753, accuracy: 0.0
Epoch 5: Error: 6.399108239574864, accuracy: 0.0
Epoch 6: Error: 6.348857761298278, accuracy: 0.0
Epoch 7: Error: 6.29879449047327, accuracy: 0.0
Epoch 8: Error: 6.248919629600734, accuracy: 0.0
Epoch 9: Error: 6.199234367886679, accuracy: 0.0
Epoch 10: Error: 6.1497398790455495, accuracy: 0.0
Epoch 11: Error: 6.100437319096966, accuracy: 0.0
Epoch 12: Error: 6.05132782416385, accuracy: 0.0
Epoch 13: Error: 6.0024125082802025, accuracy: 0.0
Epoch 14: Error: 5.953692461217287, accuracy: 0.0
Epoch 15: Error: 5.9051687463369875, accuracy: 0.0
Epoch 16: Error: 5.856842398481671, accuracy: 0.0
Epoch 17: Error: 5.808714421909786, accuracy: 0.0
Epoch 18: Error: 5.760785788286795, accuracy: 0.0
Epoch 19: Error: 5.713057434741002, accuracy: 0.0
Epoch 20: Error: 5.665530261993907, accuracy: 0.0
Epoch 21

In [12]:
import json
with open("datasets/squareData.json", "r") as f:
    a = json.load(f)
    X = a["X"]         #< 32x32 image 1 = pixel, 0 = no pixel
    y = a["y"]

print("Converting data to hilbert space")
for i in range(len(X)):
    X[i] = Mathlib.hilbertFlatten(X[i])   #< map to 1D using hilbert space (hilbert is used here so the resolution has minimal effect)
    print(f"{i}/{len(X)}", end="\r")
print(f"\rDone/{len(X)}")


try:
    with open("trainedModel.json", "r") as f:
        savedModel = json.load(f)
    spiralLayer1 = Batch(inputCount=32*32, neuronCount=32, activationFunc=Activation.LeakyReLU)
    spiralLayer2 = Batch(inputCount=32, neuronCount=2, activationFunc=Activation.Pass)

    spiralLayer1.weights = savedModel["layer1"]["weights"]
    spiralLayer1.biases = savedModel["layer1"]["biases"]
    spiralLayer2.weights = savedModel["layer2"]["weights"]
    spiralLayer2.biases = savedModel["layer2"]["biases"]
    print("Successfully loaded 'trainedModel.json'!")

except FileNotFoundError:
    print("Error: 'trainedModel.json' not found. Training from scratch.")
    spiralLayer1 = Batch(
    # spiralLayer1 = Layer(
                        inputCount=32*32,
                        neuronCount=32,   #< output features
                        activationFunc=Activation.LeakyReLU)
    spiralLayer2 = Batch(
    # spiralLayer2 = Layer(
                        inputCount=32,
                        neuronCount=2,
                        activationFunc=Activation.Pass)
softmaxCrossEntropy = Loss.SoftmaxCrossEntropy()
spiralOptimizer = Optimizer.Optimizer_SGD(learning_rate=0.3)
accuracyCalculator = Accuracy.Accuracy_hard()

for epoch in range(1_000):
    BATCH_SIZE = 32
    numSamples = len(X)

    for i in range(0, numSamples, BATCH_SIZE):
        X_batch = X[i : i + BATCH_SIZE]
        y_batch = y[i : i + BATCH_SIZE]

        ### Forwards pass
        ### layer1 -> layer2 -> layer3 -> softmax_error
        layer1Output = spiralLayer1.forward(X_batch)                              #< returns a list of lists of outputs
        layer2Output = spiralLayer2.forward(layer1Output)                         #< returns a list of lists of outputs
        # layer3Output = spiralLayer3.forward(layer2Output)                       #< returns a list of lists of outputs
        # error = softmaxCrossEntropy.forward(layer2Output, y_batch)              #< returns mean error (layer mode)
        error = softmaxCrossEntropy.forward_batch(layer2Output, y_batch)          #< returns mean error (batch mode)
        accuracy = accuracyCalculator.epoch(layer2Output, y_batch)                #< returns accuracy of model %right

        ### Backwards pass
        ### error_softmax -> layer2 ->  layer1 
        d_crossEntropy = softmaxCrossEntropy.backward_batch()
        # d_crossEntropy = softmaxCrossEntropy.backward()
        d_layer2 = spiralLayer2.backward(d_crossEntropy)
        # print([Mathlib.mean(d) for d in d_layer2])
        d_layer1 = spiralLayer1.backward(d_layer2)
        # print([Mathlib.mean(d) for d in d_layer1])

        spiralOptimizer.update(spiralLayer2)
        spiralOptimizer.update(spiralLayer1)

    # if epoch%10 == 0:
    print(f"Epoch {epoch}: Error: {error}, accuracy: {accuracy}")
    
    ## Save model
    modelState = {
        "layer1": {
            "weights": spiralLayer1.getWeights(),
            "biases": spiralLayer1.getBiases()
        },
        "layer2": {
            "weights": spiralLayer2.getWeights(),
            "biases": spiralLayer2.getBiases()
        }
    }

    with open("trainedModel.json", "w") as f:
        json.dump(modelState, f, indent=4)


Converting data to hilbert space
Done/1000
Successfully loaded 'trainedModel.json'!
Epoch 0: Error: 0.7208752008264685, accuracy: 0.498
Epoch 1: Error: 0.7204851157478904, accuracy: 0.501
Epoch 2: Error: 0.7198844772455131, accuracy: 0.5026666666666667
Epoch 3: Error: 0.7192822254216454, accuracy: 0.50375
Epoch 4: Error: 0.718677860673059, accuracy: 0.5046
Epoch 5: Error: 0.7180709698835201, accuracy: 0.5058333333333334
Epoch 6: Error: 0.7174608619890128, accuracy: 0.507
Epoch 7: Error: 0.7168463554188187, accuracy: 0.50775
Epoch 8: Error: 0.7162270242816297, accuracy: 0.5087777777777778
Epoch 9: Error: 0.7155887289783112, accuracy: 0.5101
Epoch 10: Error: 0.7149490856591993, accuracy: 0.5115454545454545
Epoch 11: Error: 0.7143136065130149, accuracy: 0.5134166666666666
Epoch 12: Error: 0.7136711226292727, accuracy: 0.5157692307692308
Epoch 13: Error: 0.7130200495047593, accuracy: 0.518
Epoch 14: Error: 0.7123600939764603, accuracy: 0.5202666666666667
Epoch 15: Error: 0.7116901038454402

The model takes slightly less than 1.5 hours to train to 77% accuracy

In [4]:
import Demo
import tkinter as tk
root = tk.Tk()
app = Demo.DrawingApp(root)
root.mainloop()